# Migrant Archive — Colab Transcription

Transcribe a YouTube video using **faster-whisper large-v3** on Colab's free T4 GPU.

### What this notebook does
1. Clones the project repo (or uses uploaded files)
2. Downloads audio from YouTube via `yt-dlp`
3. Transcribes using `faster-whisper` (large-v3, GPU)
4. Saves the result as JSON to your Google Drive

### Estimated time
| Video length | Transcription time on T4 GPU |
|-------------|------------------------------|
| 5 min | ~15 seconds |
| 30 min | ~2 minutes |
| 2 hours | ~8-10 minutes |

### After this notebook
Download the JSON from Drive → place in `data/raw/whisper/` → run `rag_test.py --rebuild`

## 1. Install Dependencies

In [ ]:
!pip install -q yt-dlp faster-whisper

print("✅ Dependencies installed")

## 2. Mount Google Drive

This is where the JSON output will be saved. You'll need to authorize access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/migrant-archive"
os.makedirs(f"{DRIVE_ROOT}/output", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/audio", exist_ok=True)

print(f"✅ Drive mounted. Output → {DRIVE_ROOT}/output/")

## 3. Get the Ingestion Code

Choose **one** method below. Option A (git clone) is recommended — it pulls all files with correct imports.

### Option A: Clone the repo (recommended)

In [ ]:
import sys
from pathlib import Path

# Clone the repo
REPO_URL = "https://github.com/basstianlopez/migrant-archive.git"
!git clone {REPO_URL} /content/migrant-archive

# Add backend/core to Python path
sys.path.insert(0, str(Path("/content/migrant-archive/backend/core")))

print("✅ Repo cloned and path configured")

### Option B: Upload files manually

If you prefer not to clone, upload these 3 files from `backend/core/`:
- `ingestion.py`
- `ingestion_audio.py`  
- `ingestion_colab.py`

Use the Files panel (left sidebar) to upload, then run this cell:

In [ ]:
# UNCOMMENT if using Option B (manual upload)
# import sys
# from pathlib import Path
# sys.path.insert(0, "/content")
# print("✅ Manual upload path configured")

## 4. Configure Your Video

Set the YouTube URL and language below. For the FILMIG channel, use `es`.
For Catalan/Spanish code-switching, try `ca` or `es`.

In [ ]:
# ══════════════════════════════════════════
# EDIT THESE TWO VALUES
# ══════════════════════════════════════════

VIDEO_URL = "https://www.youtube.com/watch?v=PASTE_VIDEO_ID_HERE"
LANGUAGE  = "es"       # es, en, ca, or "" for auto-detect

# ══════════════════════════════════════════
# Advanced (usually leave as-is)
# ══════════════════════════════════════════

MODEL     = "large-v3"  # tiny, base, small, medium, large-v3
DEVICE    = "cuda"      # cuda (GPU) or cpu

print(f"Video: {VIDEO_URL}")
print(f"Language: {LANGUAGE}  |  Model: {MODEL}  |  Device: {DEVICE}")

## 5. Run Transcription

This cell downloads the audio, runs Whisper, and saves the result.
For a 2-hour video, expect ~8-10 minutes on the free T4 GPU.

In [ ]:
import time
import json
from ingestion import VideoData, _fetch_metadata, _build_videodata, _download_audio
from ingestion_audio import _detect_device, _compute_type_for, _transcribe_audio

# ── Step 1: Fetch metadata ────────────────────────────────────────────
print("📡 Fetching video metadata ...")
info = _fetch_metadata(VIDEO_URL)
video_id = info["id"]
title = info.get("title", "Unknown")
duration_mins = info.get("duration", 0) / 60

print(f"   Title: {title}")
print(f"   Duration: {duration_mins:.0f} min")
print(f"   Channel: {info.get('channel', 'Unknown')}")

# ── Step 2: Download audio ────────────────────────────────────────────
print(f"\n⬇️  Downloading audio ...")
t0 = time.time()
audio_path = _download_audio(VIDEO_URL, output_dir=f"{DRIVE_ROOT}/audio")
print(f"   Done in {time.time() - t0:.0f}s → {audio_path}")

# ── Step 3: Transcribe ────────────────────────────────────────────────
print(f"\n🎙️  Transcribing with faster-whisper ({MODEL}, {DEVICE}) ...")
print(f"   This may take a few minutes for long videos ...")
t0 = time.time()

segments = _transcribe_audio(
    audio_path=str(audio_path),
    language=LANGUAGE,
    model_size=MODEL,
    device=DEVICE,
)

elapsed = time.time() - t0
rtf = elapsed / (info.get("duration", 1) / 60)  # real-time factor
print(f"   Done in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"   Speed: {rtf:.1f}x real-time")
print(f"   Segments: {len(segments)}")

# ── Step 4: Build VideoData & save ────────────────────────────────────
video_data = _build_videodata(info, segments)
output_path = video_data.save_json(output_dir=f"{DRIVE_ROOT}/output")

print(f"\n💾 Saved to Drive: {output_path}")
print(f"   File size: {output_path.stat().st_size / 1024:.0f} KB")

# ── Step 5: Preview ───────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"PREVIEW (first 300 chars):")
print(f"{'─'*60}")
print(video_data.full_text[:300])
print(f"{'─'*60}")

# ── Step 6: Summary ───────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"✅ TRANSCRIPTION COMPLETE")
print(f"{'='*60}")
print(f"Video:    {title}")
print(f"Duration: {duration_mins:.0f} min")
print(f"Segments: {len(segments)}")
print(f"Chars:    {len(video_data.full_text):,}")
print(f"Speed:    {rtf:.1f}x real-time")
print(f"Saved:    {output_path}")
print(f"\n📋 Next: Download this JSON → place in data/raw/whisper/ → rebuild index")

## 6. Download the Result

The JSON is saved in your Google Drive at `migrant-archive/output/{video_id}.json`.

**Option A**: Download directly from [drive.google.com](https://drive.google.com) → My Drive → migrant-archive → output

**Option B**: Run this cell to download via Colab:

In [ ]:
from google.colab import files

# Download the JSON file to your local machine
files.download(str(output_path))

print("\nPlace this file in: migrant-archive/data/raw/whisper/")
print("Then run: python backend/scripts/rag_test.py --rebuild")

---

### Troubleshooting

| Problem | Fix |
|---------|-----|
| `yt-dlp` says video unavailable | Video may be private/age-restricted. Try another URL. |
| Out of memory (OOM) | Switch to `MODEL = "medium"` — still excellent quality, less RAM. |
| GPU not available | Colab T4 quota exhausted. Wait a few hours or use `DEVICE = "cpu"` (much slower). |
| `faster-whisper` import error | Re-run the install cell (!pip install). Colab sometimes loses packages. |
| Drive mount fails | Re-run the mount cell. Make sure you're logged into the correct Google account. |